# 1 · Raster fields

The raster builders turn a pyramids `Dataset` into an interactive field: `image` (regular grid),
`quadmesh` (irregular/curvilinear cells), `contours` / `filled_contours` (iso-lines), `rgb`
(three-band composite) and `spaghetti` (one contour set per ensemble member). NoData is rendered
transparent automatically.

**Setup** — load the Bokeh extension and the bundled Lisbon DEM (a clean single-band raster, EPSG:4326).

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

from pyramids.dataset import Dataset
from digitalearth.interactive import InteractiveMap

dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))

### `image` — a pixel field
`image` draws the raster as a regular pixel grid. `cmap=None` (the default) picks a colormap from the variable via auto-style; here we ask for `terrain`.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="image")
m.image(dem, cmap="terrain")
m

### `quadmesh` — irregular cells
`quadmesh` draws every cell from its coordinate arrays, so non-uniform or curvilinear grids render without resampling. On a regular grid it matches `image`.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="quadmesh")
m.quadmesh(dem, cmap="viridis")
m

### `contours` / `filled_contours` — iso-lines
`contours` draws line iso-contours; `filled_contours` fills the bands between them. Both take a `levels` count.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="filled_contours")
m.filled_contours(dem, levels=10)
m

### `rgb` — a three-band composite
A false-colour seasonal composite of the global temperature stack: **R = January, G = July, B = December**. The ocean NoData sentinel is masked to `NaN` first so the per-band 2–98 % stretch sees only real temperatures (green = boreal summer, magenta = austral/tropical warmth, ocean transparent).

In [ ]:
import numpy as np

def _temp(month):
    a = Dataset.read_file(str(DATA / "global" / f"wc2.1_10m_tavg_{month:02d}.tif")).read_array(band=0)
    return np.where(a.astype("float32") < -1e30, np.nan, a)   # ocean sentinel -> transparent

_src = Dataset.read_file(str(DATA / "global" / "wc2.1_10m_tavg_01.tif"))
stack = np.stack([_temp(1), _temp(7), _temp(12)]).astype("float32")
composite = Dataset.create_from_array(arr=stack, geo=_src.geotransform, epsg=_src.epsg, no_data_value=np.nan)

m = InteractiveMap(crs=4326, title="seasonal RGB (Jan/Jul/Dec)")
m.rgb(composite)
m

### `spaghetti` — an ensemble of contours
Given a `DatasetCollection`, `spaghetti` overlays one contour set per member in a colour cycle — the classic 'spaghetti plot' for comparing members. Here the first four months of the temperature stack.

In [ ]:
from pyramids.dataset.collection import DatasetCollection

months = [str(DATA / "global" / f"wc2.1_10m_tavg_{m:02d}.tif") for m in range(1, 5)]
ens = DatasetCollection.from_files(months)

m = InteractiveMap(crs=4326, title="spaghetti (4 months)")
m.spaghetti(ens, levels=4)
m